# Part 5 (Work IQ 제외): Foundry IQ + Fabric IQ + Web IQ 결합하기

Zava의 여름 세일 준비 상황을 점검하기 위해 사내 정책, 제품 재고, 웹 벤치마크를 함께 조회합니다. Work IQ의 결제 정책, 테넌트 활성화 또는 앱 등록을 아직 준비하지 못했다면 이 Part에서 Work IQ 없이 통합 검색을 실습할 수 있습니다.

이 노트북은 **Foundry IQ(hrdocs + healthdocs) + Fabric IQ + Web IQ**의 3가지 IQ를 사용해 4개 지식 소스를 연결합니다.

**🎯 미션**
- HR 문서, 건강 문서(Foundry IQ), Fabric IQ, Web IQ를 결합한 **멀티 소스 지식 베이스** 구축하기
- 구조화된 제품 데이터, 사내 정책 문서, 실시간 웹 벤치마크를 아우르는 질문 실행하기
- Fabric 질문과 나머지 질문을 분리해 조회하고 결과를 함께 확인하기

## 빌딩 블록

Part 1~3에서 다룬 3가지 소스 유형의 4개 지식 소스를 재사용합니다. Work IQ를 지식 소스 목록과 질문에서 제외한 구성입니다.

- **Foundry IQ**: `hrdocs`, `healthdocs` 검색 인덱스 지식 소스
- **Fabric IQ**: Fabric 온톨로지 지식 소스 (구조화된 제품/재고 데이터)
- **Web IQ**: MCP 서버로 노출되는 실시간 웹 검색 지식 소스

## 워크플로우

<img src="../img/workflow-part5.png" width="700"/>


## 1단계: 환경 변수 설정

Azure AI Search, Azure OpenAI, Fabric, Web IQ 리소스에 대한 구성을 로드합니다. 프로젝트 폴더의 `.env` 파일에 이 파트에 필요한 값이 들어 있습니다.

‼️ `WEB_IQ_KEY`가 아직 설정되어 있지 않다면, Part 2에서 안내한 대로 `.env` 파일의 `WEB_IQ_KEY=your-web-iq-key`를 실제 키 값으로 교체해 주세요.

In [ ]:
import json
import os

from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]
WEB_IQ_KEY = os.environ["WEB_IQ_KEY"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = DefaultAzureCredential()

print("환경 변수를 불러왔습니다.")

## 2단계: Fabric 로그인

아래 셀을 실행하고 출력된 주소를 브라우저에서 엽니다. 표시된 코드를 입력한 뒤 Fabric 워크스페이스에 액세스 권한이 있는 계정으로 로그인하세요. 로그인이 완료되면 셀이 자동으로 계속됩니다.

In [ ]:
from azure.identity import DeviceCodeCredential
from work_iq_auth import device_code_prompt

user_credential = DeviceCodeCredential(tenant_id=AZURE_TENANT_ID, prompt_callback=device_code_prompt)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Fabric 로그인 완료. 인증 정보는 화면에 표시하지 않습니다.")

## 3단계: 네 개의 지식 소스 만들기

이 지식 베이스를 위해 네 개의 지식 소스를 만듭니다. Work IQ를 제외하고, 두 개의 검색 인덱스 소스(Foundry IQ)에 Fabric 온톨로지 소스와 Web IQ MCP 소스를 더합니다.

* `hrdocs-knowledge-source`: `hrdocs` 검색 인덱스를 가리킴
* `healthdocs-knowledge-source`: `healthdocs` 검색 인덱스를 가리킴
* `fabric-ontology-knowledge-source`: 구조화된 운영 데이터에 사용되는 Fabric 워크스페이스와 온톨로지를 가리킴
* `web-knowledge-source`: 키 기반 인증과 `web` 도구를 사용해 Web IQ 서비스의 원격 MCP 서버를 가리킴

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    McpServerAutoOutputParsing,
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerStoredHeadersAuthentication,
    McpServerStoredHeadersParameters,
    McpServerTool,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
WEB_KNOWLEDGE_SOURCE_NAME = "web-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
    WEB_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR 문서"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava 건강 복리후생 문서"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Zava Fabric Ontology 지식 소스",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)

web_knowledge_source = McpServerKnowledgeSource(
    name=WEB_KNOWLEDGE_SOURCE_NAME,
    description="Web IQ(실시간 웹 검색)",
    mcp_server_parameters=McpServerKnowledgeSourceParameters(
        server_url="https://api.microsoft.ai/v3/mcp",
        authentication=McpServerStoredHeadersAuthentication(
            stored_headers_parameters=McpServerStoredHeadersParameters(
                {"headers": {"x-apikey": WEB_IQ_KEY}}
            )
        ),
        tools=[McpServerTool(name="web", output_parsing=McpServerAutoOutputParsing())],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=web_knowledge_source)
print("Knowledge sources created")

## 4단계: 결합된 지식 베이스 만들기

지식 베이스는 지식 소스를 LLM 및 검색 동작 방식에 대한 지침과 결합합니다.

이 노트북에서 지식 베이스는 연결된 모델이 질문에 직접 답할 수 있도록 `answerSynthesis` `outputMode`를 사용합니다. 또한 모델이 질의 계획과 소스 선택을 도울 수 있도록 `low` `retrievalReasoningEffort`를 사용합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-foundry-fabric-web-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="HR 및 건강 문서를 위한 검색 인덱스, 제품 데이터를 위한 Fabric Ontology, 실시간 웹 검색을 위한 Web IQ를 결합한 Zava KB (Work IQ 제외)",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions=(
        "구조화된 운영 데이터에는 Fabric Ontology를, HR 및 건강 정책 문서에는 검색 인덱스를, 공개 웹 벤치마크가 필요하면 Web IQ를 사용하세요. "
        "Fabric Ontology에는 복합 질문 전체를 보내지 말고 재고 관련 하위 질문만 분리해 보내세요. "
        "Fabric Ontology 질의는 반드시 엔터티와 필드, 집계 방식을 명시한 영어 문장으로 작성하세요. "
        "예: 'Using Product entities, group by category, calculate the sum of stockLevel for each category, "
        "and return the category with the lowest total stockLevel.'"
    ),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

## 5단계: 지식 베이스에 질의

네 개의 지식 소스를 아우르는 네 부분으로 된 여름 세일 준비 점검 질문을 합니다.

* _"현재 재고 수준이 가장 낮은 제품 카테고리는 무엇인가요?"_ → **Fabric IQ**
* _"예산 관리를 담당하는 회사 직무는 무엇인가요?"_ → `hrdocs`
* _"Zava는 직원에게 어떤 정신 건강 보장을 제공하나요?"_ → `healthdocs`
* _"소매업체들의 일반적인 재고 회전율 벤치마크는 무엇인가요?"_ → **Web IQ**

ℹ️ Fabric IQ (Preview) 의 NL2Ontology 파서는 다른 주제와 섞인 복합 질문에서 간헐적으로 실패하는 경우가 있어
Fabric IQ에는 단독 질문형태로 보내고 나머지 소스는 별도 호출로 분리해 신뢰성을 확보합니다.

⏱️ Fabric IQ와 Web IQ 지식 소스의 증가된 지연 시간 때문에 샘플 질문에 답하는 데 약 1~2분이 걸립니다.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeSourceParams,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display
import time

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

fabric_question = "현재 재고 수준이 가장 낮은 제품 카테고리는 무엇인가요?"
other_sources_question = ("여름 세일 준비 점검 회의입니다: "
                           "2) 예산 관리를 담당하는 Zava 직무는 무엇인가요? "
                           "3) Zava는 직원에게 어떤 정신 건강 보장을 제공하나요? "
                           "4) 소매업체들의 일반적인 재고 회전율(inventory turnover) 벤치마크는 무엇인가요? 웹에서 최신 자료를 찾아 주세요. "
                           "답변도 2) 3) 4) 번호를 그대로 사용해 주세요.")

fabric_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=fabric_question)]),
    ],
    knowledge_source_params=[
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
            reranker_threshold=0,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

other_sources_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=other_sources_question)]),
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        KnowledgeSourceParams(
            knowledge_source_name=WEB_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            kind="mcpServer",
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=300,
)

# 단독 질문도 드물게 실패할 수 있으므로 Fabric IQ 호출만 짧게 재시도합니다.
MAX_FABRIC_ATTEMPTS = 2
fabric_result = None
fabric_failed = True
for attempt in range(1, MAX_FABRIC_ATTEMPTS + 1):
    fabric_result = knowledge_base_client.retrieve(
        retrieval_request=fabric_request,
        query_source_authorization=user_token,
    )
    fabric_activities = [activity.as_dict() for activity in fabric_result.activity or []
                         if getattr(activity, "type", None) == "fabricOntology"]
    fabric_references = [ref for ref in fabric_result.references or []
                         if getattr(ref, "type", None) == "fabricOntology"]
    fabric_failed = not fabric_activities or any(activity.get("error") for activity in fabric_activities) or not fabric_references
    if not fabric_failed:
        break
    print(json.dumps(fabric_activities, indent=2, ensure_ascii=False))
    if attempt < MAX_FABRIC_ATTEMPTS:
        print(f"⚠️ Fabric IQ 질의가 실패했습니다 (시도 {attempt}/{MAX_FABRIC_ATTEMPTS}). 재시도 중...")
        time.sleep(3)
if fabric_failed:
    raise RuntimeError("Fabric IQ 검증 실패. 활동 로그와 Part 3의 Troubleshooting을 확인하세요. 다른 소스의 답변만으로 성공으로 판단하지 않습니다.")

other_result = knowledge_base_client.retrieve(
    retrieval_request=other_sources_request,
    query_source_authorization=user_token,
)

# 활동 로그/참조 검토 셀에서 두 호출 결과를 모두 볼 수 있도록 하나로 병합
result = other_result
result.activity = (fabric_result.activity or []) + (other_result.activity or [])
result.references = (fabric_result.references or []) + (other_result.references or [])

source_errors = [activity.as_dict() for activity in result.activity or [] if getattr(activity, "error", None)]
if source_errors:
    print(json.dumps(source_errors, indent=2, ensure_ascii=False))
    raise RuntimeError("지식 소스 조회 실패. Web IQ 키와 활동 로그를 확인하세요. 일부 소스의 답변만으로 성공으로 판단하지 않습니다.")
if not any(getattr(ref, "type", None) == "mcpServer" for ref in result.references or []):
    raise RuntimeError("Web IQ 참조가 없습니다. 웹 검색 결과와 활동 로그를 확인하세요.")

fabric_answer = fabric_result.response[0].content[0].text
answer_text = f"1) {fabric_answer}\n\n" + other_result.response[0].content[0].text

# 응답 메시지에서 합성된 답변 추출
display(Markdown(answer_text))


### 활동 로그 검토

이 활동 로그에서는 "fabricOntology", "mcpServer", "searchIndex"에 대한 호출과 각 단계의 질의를 볼 수 있습니다.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2, ensure_ascii=False))

### 참조 검토

지식 베이스의 여러 소스에서 온 다양한 참조 유형이 혼합되어 보일 것입니다.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2, ensure_ascii=False))

#### 🔍 소스 헌트

위에 출력된 참조를 살펴보세요. 3가지 소스 유형(searchIndex, fabricOntology, mcpServer)과 4개 지식 소스를 식별할 수 있나요?

1. 어떤 참조가 **Fabric IQ**(구조화된 제품 엔터티)에서 왔나요?
2. 어떤 참조가 **HR 인덱스**(정책 문서)에서 왔나요?
3. 어떤 참조가 **healthdocs**(건강 복리후생 문서)에서 왔나요?
4. 어떤 참조가 **Web IQ**(실시간 웹 검색)에서 왔나요?

## ✅ 미션 완료

**무엇을 만들었나:**

* ✓ **Work IQ 없는 4개 소스 지식 베이스**: HR 문서, 건강 문서, 구조화된 Fabric 제품 데이터, 실시간 웹 검색을 연결하며 Work IQ 인증과 결제 설정 없이 실습할 수 있음.
* ✓ **분리 호출 기반 멀티 소스 검색**: Fabric 질문은 단독 조회하고 나머지 질문은 문서와 웹 소스로 라우팅한 뒤, 두 호출의 활동 로그와 참조를 함께 확인함.
* ✓ **Foundry IQ + Fabric IQ + Web IQ 조합**: Fabric 답변과 문서 및 웹 답변을 인용과 함께 한 화면에 표시함.

➡️ Work IQ 설정을 완료했다면 [Part 6: 모든 소스를 한데 모으기 (Work IQ 포함)](part6-work-iq-fabric-iq-to-kb.ipynb)에서 Work IQ까지 포함한 5개 지식 소스 구성을 확인해 보세요.